In [1]:
from pyspark.sql import SparkSession

# Forzamos cierre de sesiones previas con versiones erróneas
try: spark.stop()
except: pass

spark = SparkSession.builder \
    .appName("Analisis_BigData_Final") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/TiendaBigData.AmazonLaptops") \
    .getOrCreate()

df = spark.read.format("mongodb").load()
print(f"Éxito total: {df.count()} registros.")
df.show(5)

Éxito total: 97 registros.
+--------------------+-------------------+-------+--------------------+-----+
|                 _id|      fecha_captura|  grupo|       identificador|valor|
+--------------------+-------------------+-------+--------------------+-----+
|69eadb58f62d61430...|2026-04-24 02:53:33|YHadfeg|Laptop | Laptops ...|450.0|
|69eadb58f62d61430...|2026-04-24 02:53:33|YHadfeg|CYCLUS 2025 15.6 ...|499.0|
|69eadb58f62d61430...|2026-04-24 02:53:33|YHadfeg|Laptop 15.6 Pouce...|313.0|
|69eadb58f62d61430...|2026-04-24 02:53:33|YHadfeg|acer Aspire Go 15...|469.0|
|69eadb58f62d61430...|2026-04-24 02:53:33|YHadfeg|Laptop 14 inch Du...|209.0|
+--------------------+-------------------+-------+--------------------+-----+
only showing top 5 rows



In [2]:
from pyspark.sql.functions import col, split, when, avg
# 2. Transformación de Negocio: Extraer la MARCA
# En Amazon, la primera palabra del título suele ser la marca.
df_transformado = df.withColumn("marca", split(col("identificador"), " ")[0])

# 3. Limpieza de Outliers: Filtrar laptops con precios erróneos (ej: < 100 euros)
df_validado = df_transformado.filter(col("valor") > 100)

# 4. Agregación: Precio promedio por Marca
reporte_marcas = df_validado.groupBy("marca").agg(
    avg("valor").alias("precio_promedio")
).orderBy(col("precio_promedio").desc())

reporte_marcas.show()

+---------+------------------+
|    marca|   precio_promedio|
+---------+------------------+
|       LG|            1474.0|
|Microsoft|            1049.0|
|   Lenovo| 811.9545454545455|
|     ASUS|          784.1875|
|     DELL|             749.0|
|     Acer|             690.5|
| ACEMAGIC| 682.3333333333334|
|     Dell|             600.5|
|   CYCLUS| 600.3333333333334|
|  Lenovo,|             579.0|
|      MSI|             569.0|
|     acer|             556.0|
|Blackview|             479.0|
|     2021|             390.0|
|       HP|             366.4|
|   Laptop|357.44444444444446|
|     15.6|             336.5|
|    15.6"|             319.0|
|   KEFEYA|             309.0|
|    Ryzen|             299.0|
+---------+------------------+
only showing top 20 rows

